In [ ]:
import os
import json
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =========================================================
# 1. CONFIG
# =========================================================
DATASET_PATH = "P1"          # change if needed
SPLIT_NAME = "P1S1"          # "P1S1" or "P1S2"
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
NUM_WORKERS = 0
SEED = 42
IMG_W, IMG_H = 640, 480

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("====================================")
print("MMVR Human Pose Project")
print(f"Device      : {DEVICE}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Split       : {SPLIT_NAME}")
print("====================================")

# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================================================
# 3. CONSTANTS / HELPERS
# =========================================================
SKELETON_CONNECTIONS = np.array([
    [13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
    [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
    [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
    [3, 5], [4, 6]
], dtype=np.int32)

OKS_SIGMAS = np.array([
    0.026, 0.025, 0.025, 0.035, 0.035,
    0.079, 0.079, 0.072, 0.072, 0.062,
    0.062, 0.107, 0.107, 0.087, 0.087,
    0.089, 0.089
], dtype=np.float32)

def normalize_heatmap(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-6)

def get_relative_path(root_dir, path):
    return os.path.relpath(path, root_dir).replace("\\", "/")

def flatten_split_entries(split_entries):
    out = set()
    if isinstance(split_entries, np.ndarray):
        split_entries = split_entries.tolist()
    for item in split_entries:
        if isinstance(item, bytes):
            item = item.decode("utf-8")
        item = str(item).replace("\\", "/").strip("/")
        out.add(item)
    return out

def masked_mse_loss(pred, gt, valid_mask):
    mask = valid_mask.unsqueeze(-1).float()
    loss = ((pred - gt) ** 2 * mask).sum() / (mask.sum() + 1e-8)
    return loss

def pck_counts(pred, gt, valid_mask, threshold=0.10):
    d = torch.norm(pred - gt, dim=2)
    correct = ((d < threshold) & valid_mask).sum().item()
    total = valid_mask.sum().item()
    return correct, total

def compute_pck(correct, total):
    return 100.0 * correct / total if total > 0 else 0.0

def oks_per_instance(pred_xy, gt_xy, valid_mask, bbox_area, sigmas=OKS_SIGMAS):
    valid = valid_mask.astype(bool)
    if valid.sum() == 0:
        return 0.0
    d2 = np.sum((pred_xy - gt_xy) ** 2, axis=1)
    denom = 2 * (sigmas ** 2) * (bbox_area + 1e-6)
    oks = np.exp(-d2 / (denom + 1e-9))
    return float(np.mean(oks[valid]))

def depth_from_bbox_hori(bbox_hori):
    if bbox_hori is None or len(bbox_hori) == 0:
        return 0.0
    x1, _, x2, _ = bbox_hori[0]
    return float(0.5 * (x1 + x2))

# =========================================================
# 4. DATASET
# =========================================================
class RadarPoseDataset(Dataset):
    def __init__(self, root_dir, view="both", allowed_segments=None):
        self.root_dir = root_dir
        self.view = view
        self.allowed_segments = allowed_segments
        self.samples = []
        self._build_index()

    def _build_index(self):
        for root, _, files in os.walk(self.root_dir):
            rel_dir = get_relative_path(self.root_dir, root)

            if self.allowed_segments is not None:
                if rel_dir not in self.allowed_segments:
                    continue

            radar_files = sorted([f for f in files if f.endswith("_radar.npz")])
            for f in radar_files:
                base = f.replace("_radar.npz", "")
                radar_path = os.path.join(root, f)
                pose_path = os.path.join(root, f"{base}_pose.npz")
                bbox_path = os.path.join(root, f"{base}_bbox.npz")
                meta_path = os.path.join(root, f"{base}_meta.npz")

                if os.path.exists(pose_path) and os.path.exists(bbox_path):
                    self.samples.append({
                        "radar_path": radar_path,
                        "pose_path": pose_path,
                        "bbox_path": bbox_path,
                        "meta_path": meta_path if os.path.exists(meta_path) else None,
                        "segment": rel_dir,
                        "frame_id": base
                    })

        self.samples.sort(key=lambda x: (x["segment"], x["frame_id"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        radar_data = np.load(s["radar_path"])
        hori = normalize_heatmap(radar_data["hm_hori"])
        vert = normalize_heatmap(radar_data["hm_vert"])

        if self.view == "both":
            radar = np.stack([hori, vert], axis=0)
        elif self.view == "hori":
            radar = np.expand_dims(hori, axis=0)
        else:
            raise ValueError("view must be 'both' or 'hori'")

        pose_data = np.load(s["pose_path"])
        kp_all = pose_data["kp"]  # (n,17,3)

        # P1 is single-subject
        kp = kp_all[0].astype(np.float32)
        kp_xy = kp[:, :2].copy()
        kp_score = kp[:, 2].copy()

        valid = (kp_score > 0) & ~((kp[:, 0] == 0) & (kp[:, 1] == 0))

        kp_norm = kp_xy.copy()
        kp_norm[:, 0] /= IMG_W
        kp_norm[:, 1] /= IMG_H

        bbox_data = np.load(s["bbox_path"])
        bbox_i = bbox_data["bbox_i"]
        bbox_hori = bbox_data["bbox_hori"]

        if len(bbox_i) > 0:
            x1, y1, x2, y2 = bbox_i[0, :4]
            bbox_area = max((x2 - x1) * (y2 - y1), 1.0)
        else:
            bbox_area = 1.0

        depth_target = depth_from_bbox_hori(bbox_hori)

        global_frame_id = -1
        if s["meta_path"] is not None:
            meta = np.load(s["meta_path"])
            if "global_frame_id" in meta:
                global_frame_id = int(meta["global_frame_id"])

        return {
            "radar": torch.tensor(radar, dtype=torch.float32),
            "keypoints": torch.tensor(kp_norm, dtype=torch.float32),
            "valid": torch.tensor(valid, dtype=torch.bool),
            "bbox_area": torch.tensor(bbox_area, dtype=torch.float32),
            "depth_target": torch.tensor(depth_target, dtype=torch.float32),
            "segment": s["segment"],
            "frame_id": s["frame_id"],
            "global_frame_id": global_frame_id
        }

# =========================================================
# 5. MODEL
# =========================================================
class RadarPoseDepthModel(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.shared_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.pose_head = nn.Linear(512, 17 * 2)
        self.depth_head = nn.Linear(512, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.shared_fc(feat)

        pose = torch.sigmoid(self.pose_head(feat)).view(-1, 17, 2)
        depth = self.depth_head(feat).squeeze(1)

        return pose, depth

# =========================================================
# 6. SPLITS
# =========================================================
def load_split(dataset_path, split_name="P1S1"):
    split_file = os.path.join(dataset_path, "data_split.npz")
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    data = np.load(split_file, allow_pickle=True)
    split_dict = data["data_split_dict"].item()

    if split_name not in split_dict:
        raise ValueError(f"{split_name} not found in split file")

    info = split_dict[split_name]
    train_segments = flatten_split_entries(info["train"])
    val_segments = flatten_split_entries(info["val"])
    test_segments = flatten_split_entries(info["test"])
    return train_segments, val_segments, test_segments

# =========================================================
# 7. TRAIN / EVAL
# =========================================================
def run_epoch(model, loader, optimizer=None, depth_weight=0.1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_pose_loss = 0.0
    total_depth_loss = 0.0

    total_correct = 0
    total_valid = 0
    oks_scores = []

    depth_abs_errors = []

    for batch in loader:
        radar = batch["radar"].to(DEVICE)
        gt_kp = batch["keypoints"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        gt_depth = batch["depth_target"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred_kp, pred_depth = model(radar)

            pose_loss = masked_mse_loss(pred_kp, gt_kp, valid)
            depth_loss = nn.functional.l1_loss(pred_depth, gt_depth)
            loss = pose_loss + depth_weight * depth_loss

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        total_pose_loss += pose_loss.item()
        total_depth_loss += depth_loss.item()

        c, t = pck_counts(pred_kp.detach(), gt_kp.detach(), valid.detach(), threshold=0.10)
        total_correct += c
        total_valid += t

        pred_np = pred_kp.detach().cpu().numpy()
        gt_np = gt_kp.detach().cpu().numpy()
        valid_np = valid.detach().cpu().numpy()
        bbox_area_np = batch["bbox_area"].cpu().numpy()

        pred_depth_np = pred_depth.detach().cpu().numpy()
        gt_depth_np = gt_depth.detach().cpu().numpy()
        depth_abs_errors.extend(np.abs(pred_depth_np - gt_depth_np).tolist())

        for i in range(len(pred_np)):
            pred_px = pred_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            gt_px = gt_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            oks_scores.append(
                oks_per_instance(pred_px, gt_px, valid_np[i], float(bbox_area_np[i]))
            )

    avg_loss = total_loss / max(len(loader), 1)
    avg_pose_loss = total_pose_loss / max(len(loader), 1)
    avg_depth_loss = total_depth_loss / max(len(loader), 1)
    avg_pck = compute_pck(total_correct, total_valid)
    avg_oks = float(np.mean(oks_scores)) if oks_scores else 0.0
    depth_mae = float(np.mean(depth_abs_errors)) if depth_abs_errors else 0.0

    return {
        "loss": avg_loss,
        "pose_loss": avg_pose_loss,
        "depth_loss": avg_depth_loss,
        "pck": avg_pck,
        "oks": avg_oks,
        "depth_mae": depth_mae
    }

# =========================================================
# 8. EXPERIMENT RUNNER
# =========================================================
def train_and_evaluate(model_name, view):
    print(f"\n================ {model_name} ================")

    train_segments, val_segments, test_segments = load_split(DATASET_PATH, SPLIT_NAME)

    train_dataset = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=train_segments)
    val_dataset   = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=val_segments)
    test_dataset  = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=test_segments)

    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples  : {len(val_dataset)}")
    print(f"Test samples : {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    in_channels = 2 if view == "both" else 1
    model = RadarPoseDepthModel(in_channels=in_channels).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {
        "train": [],
        "val": []
    }

    best_val_pck = -1.0
    best_path = f"best_{model_name.lower()}_{SPLIT_NAME.lower()}.pth"

    for epoch in range(EPOCHS):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)

        history["train"].append(train_metrics)
        history["val"].append(val_metrics)

        if val_metrics["pck"] > best_val_pck:
            best_val_pck = val_metrics["pck"]
            torch.save(model.state_dict(), best_path)

        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train PCK: {train_metrics['pck']:.2f}% | "
            f"Val PCK: {val_metrics['pck']:.2f}% | "
            f"Val OKS: {val_metrics['oks']:.4f} | "
            f"Val Depth MAE: {val_metrics['depth_mae']:.4f}"
        )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_metrics = run_epoch(model, test_loader, optimizer=None)

    print(f"\nFinal Test Metrics for {model_name}")
    print(json.dumps(test_metrics, indent=2))

    hist_path = f"history_{model_name.lower()}_{SPLIT_NAME.lower()}.json"
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)

    return {
        "model_name": model_name,
        "view": view,
        "best_model_path": best_path,
        "history": history,
        "test_metrics": test_metrics,
        "test_dataset": test_dataset,
        "model": model
    }

# =========================================================
# 9. RUN BOTH MODELS AUTOMATICALLY
# =========================================================
results_baseline = train_and_evaluate("BASELINE", view="hori")
results_fusion = train_and_evaluate("FUSION", view="both")

# =========================================================
# 10. FINAL COMPARISON TABLE
# =========================================================
print("\n================ FINAL COMPARISON ================")
print(f"{'Model':<12} {'PCK (%)':<10} {'OKS':<10} {'Depth MAE':<12} {'Loss':<10}")
for res in [results_baseline, results_fusion]:
    m = res["test_metrics"]
    print(f"{res['model_name']:<12} {m['pck']:<10.2f} {m['oks']:<10.4f} {m['depth_mae']:<12.4f} {m['loss']:<10.4f}")

# =========================================================
# 11. PLOT TRAINING CURVES
# =========================================================
def plot_history(history, title_prefix):
    train_pck = [x["pck"] for x in history["train"]]
    val_pck = [x["pck"] for x in history["val"]]
    train_oks = [x["oks"] for x in history["train"]]
    val_oks = [x["oks"] for x in history["val"]]
    train_loss = [x["loss"] for x in history["train"]]
    val_loss = [x["loss"] for x in history["val"]]

    plt.figure(figsize=(8, 4))
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(train_pck, label="Train PCK")
    plt.plot(val_pck, label="Val PCK")
    plt.title(f"{title_prefix} PCK")
    plt.xlabel("Epoch")
    plt.ylabel("PCK (%)")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(train_oks, label="Train OKS")
    plt.plot(val_oks, label="Val OKS")
    plt.title(f"{title_prefix} OKS")
    plt.xlabel("Epoch")
    plt.ylabel("OKS")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(results_baseline["history"], "BASELINE")
plot_history(results_fusion["history"], "FUSION")

# =========================================================
# 12. VISUALIZATION
# =========================================================
def draw_pose(ax, kp_px, valid, color, label=None):
    for c in SKELETON_CONNECTIONS:
        i, j = c
        if valid[i] and valid[j]:
            p1, p2 = kp_px[i], kp_px[j]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, linewidth=2)
    pts = kp_px[valid]
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=20, label=label)

def show_predictions(result_obj, num_samples=3):
    model = result_obj["model"]
    dataset = result_obj["test_dataset"]
    model.eval()

    n = min(num_samples, len(dataset))
    for idx in range(n):
        sample = dataset[idx]
        radar = sample["radar"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_kp, pred_depth = model(radar)

        pred_kp = pred_kp.cpu().numpy()[0]
        gt_kp = sample["keypoints"].numpy()
        valid = sample["valid"].numpy()

        pred_px = pred_kp * np.array([IMG_W, IMG_H], dtype=np.float32)
        gt_px = gt_kp * np.array([IMG_W, IMG_H], dtype=np.float32)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")

        if sample["radar"].shape[0] == 2:
            axes[1].imshow(sample["radar"][1].numpy())
            axes[1].set_title("Vertical Radar")
        else:
            axes[1].axis("off")

        canvas = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
        axes[2].imshow(canvas)
        draw_pose(axes[2], gt_px, valid, color="g", label="GT")
        draw_pose(axes[2], pred_px, valid, color="r", label="Pred")
        axes[2].set_title(
            f"{result_obj['model_name']} | "
            f"Pred Depth={float(pred_depth.cpu().numpy()[0]):.2f}"
        )
        axes[2].legend()

        plt.tight_layout()
        plt.show()

show_predictions(results_baseline, num_samples=3)
show_predictions(results_fusion, num_samples=3)

In [1]:
import os
import json
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =========================================================
# 1. CONFIG
# =========================================================
DATASET_PATH = "P1"          # change if needed
SPLIT_NAME = "P1S1"          # "P1S1" or "P1S2"
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
NUM_WORKERS = 0
SEED = 42
IMG_W, IMG_H = 640, 480

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("====================================")
print("MMVR Human Pose Project")
print(f"Device      : {DEVICE}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Split       : {SPLIT_NAME}")
print("====================================")

# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================================================
# 3. CONSTANTS / HELPERS
# =========================================================
SKELETON_CONNECTIONS = np.array([
    [13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
    [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
    [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
    [3, 5], [4, 6]
], dtype=np.int32)

OKS_SIGMAS = np.array([
    0.026, 0.025, 0.025, 0.035, 0.035,
    0.079, 0.079, 0.072, 0.072, 0.062,
    0.062, 0.107, 0.107, 0.087, 0.087,
    0.089, 0.089
], dtype=np.float32)

def normalize_heatmap(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-6)

def get_relative_path(root_dir, path):
    return os.path.relpath(path, root_dir).replace("\\", "/")

def flatten_split_entries(split_entries):
    out = set()
    if isinstance(split_entries, np.ndarray):
        split_entries = split_entries.tolist()
    for item in split_entries:
        if isinstance(item, bytes):
            item = item.decode("utf-8")
        item = str(item).replace("\\", "/").strip("/")
        out.add(item)
    return out

def masked_mse_loss(pred, gt, valid_mask):
    mask = valid_mask.unsqueeze(-1).float()
    loss = ((pred - gt) ** 2 * mask).sum() / (mask.sum() + 1e-8)
    return loss

def pck_counts(pred, gt, valid_mask, threshold=0.10):
    d = torch.norm(pred - gt, dim=2)
    correct = ((d < threshold) & valid_mask).sum().item()
    total = valid_mask.sum().item()
    return correct, total

def compute_pck(correct, total):
    return 100.0 * correct / total if total > 0 else 0.0

def oks_per_instance(pred_xy, gt_xy, valid_mask, bbox_area, sigmas=OKS_SIGMAS):
    valid = valid_mask.astype(bool)
    if valid.sum() == 0:
        return 0.0
    d2 = np.sum((pred_xy - gt_xy) ** 2, axis=1)
    denom = 2 * (sigmas ** 2) * (bbox_area + 1e-6)
    oks = np.exp(-d2 / (denom + 1e-9))
    return float(np.mean(oks[valid]))

def depth_from_bbox_hori(bbox_hori):
    if bbox_hori is None or len(bbox_hori) == 0:
        return 0.0
    x1, _, x2, _ = bbox_hori[0]
    return float(0.5 * (x1 + x2))

# =========================================================
# 4. DATASET
# =========================================================
class RadarPoseDataset(Dataset):
    def __init__(self, root_dir, view="both", allowed_segments=None):
        self.root_dir = root_dir
        self.view = view
        self.allowed_segments = allowed_segments
        self.samples = []
        self._build_index()

    def _build_index(self):
        for root, _, files in os.walk(self.root_dir):
            rel_dir = get_relative_path(self.root_dir, root)

            if self.allowed_segments is not None:
                if rel_dir not in self.allowed_segments:
                    continue

            radar_files = sorted([f for f in files if f.endswith("_radar.npz")])
            for f in radar_files:
                base = f.replace("_radar.npz", "")
                radar_path = os.path.join(root, f)
                pose_path = os.path.join(root, f"{base}_pose.npz")
                bbox_path = os.path.join(root, f"{base}_bbox.npz")
                meta_path = os.path.join(root, f"{base}_meta.npz")

                if os.path.exists(pose_path) and os.path.exists(bbox_path):
                    self.samples.append({
                        "radar_path": radar_path,
                        "pose_path": pose_path,
                        "bbox_path": bbox_path,
                        "meta_path": meta_path if os.path.exists(meta_path) else None,
                        "segment": rel_dir,
                        "frame_id": base
                    })

        self.samples.sort(key=lambda x: (x["segment"], x["frame_id"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        radar_data = np.load(s["radar_path"])
        hori = normalize_heatmap(radar_data["hm_hori"])
        vert = normalize_heatmap(radar_data["hm_vert"])

        if self.view == "both":
            radar = np.stack([hori, vert], axis=0)
        elif self.view == "hori":
            radar = np.expand_dims(hori, axis=0)
        else:
            raise ValueError("view must be 'both' or 'hori'")

        pose_data = np.load(s["pose_path"])
        kp_all = pose_data["kp"]  # (n,17,3)

        # P1 is single-subject
        kp = kp_all[0].astype(np.float32)
        kp_xy = kp[:, :2].copy()
        kp_score = kp[:, 2].copy()

        valid = (kp_score > 0) & ~((kp[:, 0] == 0) & (kp[:, 1] == 0))

        kp_norm = kp_xy.copy()
        kp_norm[:, 0] /= IMG_W
        kp_norm[:, 1] /= IMG_H

        bbox_data = np.load(s["bbox_path"])
        bbox_i = bbox_data["bbox_i"]
        bbox_hori = bbox_data["bbox_hori"]

        if len(bbox_i) > 0:
            x1, y1, x2, y2 = bbox_i[0, :4]
            bbox_area = max((x2 - x1) * (y2 - y1), 1.0)
        else:
            bbox_area = 1.0

        depth_target = depth_from_bbox_hori(bbox_hori)

        global_frame_id = -1
        if s["meta_path"] is not None:
            meta = np.load(s["meta_path"])
            if "global_frame_id" in meta:
                global_frame_id = int(meta["global_frame_id"])

        return {
            "radar": torch.tensor(radar, dtype=torch.float32),
            "keypoints": torch.tensor(kp_norm, dtype=torch.float32),
            "valid": torch.tensor(valid, dtype=torch.bool),
            "bbox_area": torch.tensor(bbox_area, dtype=torch.float32),
            "depth_target": torch.tensor(depth_target, dtype=torch.float32),
            "segment": s["segment"],
            "frame_id": s["frame_id"],
            "global_frame_id": global_frame_id
        }

# =========================================================
# 5. MODEL
# =========================================================
class RadarPoseDepthModel(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.shared_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.pose_head = nn.Linear(512, 17 * 2)
        self.depth_head = nn.Linear(512, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.shared_fc(feat)

        pose = torch.sigmoid(self.pose_head(feat)).view(-1, 17, 2)
        depth = self.depth_head(feat).squeeze(1)

        return pose, depth

# =========================================================
# 6. SPLITS
# =========================================================
def load_split(dataset_path, split_name="P1S1"):
    split_file = os.path.join(dataset_path, "data_split.npz")
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    data = np.load(split_file, allow_pickle=True)
    split_dict = data["data_split_dict"].item()

    if split_name not in split_dict:
        raise ValueError(f"{split_name} not found in split file")

    info = split_dict[split_name]
    train_segments = flatten_split_entries(info["train"])
    val_segments = flatten_split_entries(info["val"])
    test_segments = flatten_split_entries(info["test"])
    return train_segments, val_segments, test_segments

# =========================================================
# 7. TRAIN / EVAL
# =========================================================
def run_epoch(model, loader, optimizer=None, depth_weight=0.1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_pose_loss = 0.0
    total_depth_loss = 0.0

    total_correct = 0
    total_valid = 0
    oks_scores = []

    depth_abs_errors = []

    for batch in loader:
        radar = batch["radar"].to(DEVICE)
        gt_kp = batch["keypoints"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        gt_depth = batch["depth_target"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred_kp, pred_depth = model(radar)

            pose_loss = masked_mse_loss(pred_kp, gt_kp, valid)
            depth_loss = nn.functional.l1_loss(pred_depth, gt_depth)
            loss = pose_loss + depth_weight * depth_loss

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        total_pose_loss += pose_loss.item()
        total_depth_loss += depth_loss.item()

        c, t = pck_counts(pred_kp.detach(), gt_kp.detach(), valid.detach(), threshold=0.10)
        total_correct += c
        total_valid += t

        pred_np = pred_kp.detach().cpu().numpy()
        gt_np = gt_kp.detach().cpu().numpy()
        valid_np = valid.detach().cpu().numpy()
        bbox_area_np = batch["bbox_area"].cpu().numpy()

        pred_depth_np = pred_depth.detach().cpu().numpy()
        gt_depth_np = gt_depth.detach().cpu().numpy()
        depth_abs_errors.extend(np.abs(pred_depth_np - gt_depth_np).tolist())

        for i in range(len(pred_np)):
            pred_px = pred_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            gt_px = gt_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            oks_scores.append(
                oks_per_instance(pred_px, gt_px, valid_np[i], float(bbox_area_np[i]))
            )

    avg_loss = total_loss / max(len(loader), 1)
    avg_pose_loss = total_pose_loss / max(len(loader), 1)
    avg_depth_loss = total_depth_loss / max(len(loader), 1)
    avg_pck = compute_pck(total_correct, total_valid)
    avg_oks = float(np.mean(oks_scores)) if oks_scores else 0.0
    depth_mae = float(np.mean(depth_abs_errors)) if depth_abs_errors else 0.0

    return {
        "loss": avg_loss,
        "pose_loss": avg_pose_loss,
        "depth_loss": avg_depth_loss,
        "pck": avg_pck,
        "oks": avg_oks,
        "depth_mae": depth_mae
    }

# =========================================================
# 8. EXPERIMENT RUNNER
# =========================================================
def train_and_evaluate(model_name, view):
    print(f"\n================ {model_name} ================")

    train_segments, val_segments, test_segments = load_split(DATASET_PATH, SPLIT_NAME)

    train_dataset = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=train_segments)
    val_dataset   = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=val_segments)
    test_dataset  = RadarPoseDataset(DATASET_PATH, view=view, allowed_segments=test_segments)

    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples  : {len(val_dataset)}")
    print(f"Test samples : {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    in_channels = 2 if view == "both" else 1
    model = RadarPoseDepthModel(in_channels=in_channels).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {
        "train": [],
        "val": []
    }

    best_val_pck = -1.0
    best_path = f"best_{model_name.lower()}_{SPLIT_NAME.lower()}.pth"

    for epoch in range(EPOCHS):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)

        history["train"].append(train_metrics)
        history["val"].append(val_metrics)

        if val_metrics["pck"] > best_val_pck:
            best_val_pck = val_metrics["pck"]
            torch.save(model.state_dict(), best_path)

        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train PCK: {train_metrics['pck']:.2f}% | "
            f"Val PCK: {val_metrics['pck']:.2f}% | "
            f"Val OKS: {val_metrics['oks']:.4f} | "
            f"Val Depth MAE: {val_metrics['depth_mae']:.4f}"
        )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_metrics = run_epoch(model, test_loader, optimizer=None)

    print(f"\nFinal Test Metrics for {model_name}")
    print(json.dumps(test_metrics, indent=2))

    hist_path = f"history_{model_name.lower()}_{SPLIT_NAME.lower()}.json"
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)

    return {
        "model_name": model_name,
        "view": view,
        "best_model_path": best_path,
        "history": history,
        "test_metrics": test_metrics,
        "test_dataset": test_dataset,
        "model": model
    }

# =========================================================
# 9. RUN BOTH MODELS AUTOMATICALLY
# =========================================================
results_baseline = train_and_evaluate("BASELINE", view="hori")
results_fusion = train_and_evaluate("FUSION", view="both")

# =========================================================
# 10. FINAL COMPARISON TABLE
# =========================================================
print("\n================ FINAL COMPARISON ================")
print(f"{'Model':<12} {'PCK (%)':<10} {'OKS':<10} {'Depth MAE':<12} {'Loss':<10}")
for res in [results_baseline, results_fusion]:
    m = res["test_metrics"]
    print(f"{res['model_name']:<12} {m['pck']:<10.2f} {m['oks']:<10.4f} {m['depth_mae']:<12.4f} {m['loss']:<10.4f}")

# =========================================================
# 11. PLOT TRAINING CURVES
# =========================================================
def plot_history(history, title_prefix):
    train_pck = [x["pck"] for x in history["train"]]
    val_pck = [x["pck"] for x in history["val"]]
    train_oks = [x["oks"] for x in history["train"]]
    val_oks = [x["oks"] for x in history["val"]]
    train_loss = [x["loss"] for x in history["train"]]
    val_loss = [x["loss"] for x in history["val"]]

    plt.figure(figsize=(8, 4))
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(train_pck, label="Train PCK")
    plt.plot(val_pck, label="Val PCK")
    plt.title(f"{title_prefix} PCK")
    plt.xlabel("Epoch")
    plt.ylabel("PCK (%)")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(train_oks, label="Train OKS")
    plt.plot(val_oks, label="Val OKS")
    plt.title(f"{title_prefix} OKS")
    plt.xlabel("Epoch")
    plt.ylabel("OKS")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(results_baseline["history"], "BASELINE")
plot_history(results_fusion["history"], "FUSION")

# =========================================================
# 12. VISUALIZATION
# =========================================================
def draw_pose(ax, kp_px, valid, color, label=None):
    for c in SKELETON_CONNECTIONS:
        i, j = c
        if valid[i] and valid[j]:
            p1, p2 = kp_px[i], kp_px[j]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, linewidth=2)
    pts = kp_px[valid]
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=20, label=label)

def show_predictions(result_obj, num_samples=3):
    model = result_obj["model"]
    dataset = result_obj["test_dataset"]
    model.eval()

    n = min(num_samples, len(dataset))
    for idx in range(n):
        sample = dataset[idx]
        radar = sample["radar"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_kp, pred_depth = model(radar)

        pred_kp = pred_kp.cpu().numpy()[0]
        gt_kp = sample["keypoints"].numpy()
        valid = sample["valid"].numpy()

        pred_px = pred_kp * np.array([IMG_W, IMG_H], dtype=np.float32)
        gt_px = gt_kp * np.array([IMG_W, IMG_H], dtype=np.float32)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")

        if sample["radar"].shape[0] == 2:
            axes[1].imshow(sample["radar"][1].numpy())
            axes[1].set_title("Vertical Radar")
        else:
            axes[1].axis("off")

        canvas = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
        axes[2].imshow(canvas)
        draw_pose(axes[2], gt_px, valid, color="g", label="GT")
        draw_pose(axes[2], pred_px, valid, color="r", label="Pred")
        axes[2].set_title(
            f"{result_obj['model_name']} | "
            f"Pred Depth={float(pred_depth.cpu().numpy()[0]):.2f}"
        )
        axes[2].legend()

        plt.tight_layout()
        plt.show()

show_predictions(results_baseline, num_samples=3)
show_predictions(results_fusion, num_samples=3)

Matplotlib is building the font cache; this may take a moment.


MMVR Human Pose Project
Device      : mps
Dataset Path: P1
Split       : P1S1

================ BASELINE ================
Train samples: 86579
Val samples  : 10538
Test samples : 10785
Epoch [01/15] | Train PCK: 68.85% | Val PCK: 81.31% | Val OKS: 0.1786 | Val Depth MAE: 2.9036
Epoch [02/15] | Train PCK: 76.83% | Val PCK: 85.44% | Val OKS: 0.1812 | Val Depth MAE: 1.9197
Epoch [03/15] | Train PCK: 79.56% | Val PCK: 85.10% | Val OKS: 0.1560 | Val Depth MAE: 2.0503
Epoch [04/15] | Train PCK: 81.93% | Val PCK: 89.12% | Val OKS: 0.2504 | Val Depth MAE: 1.9352
Epoch [05/15] | Train PCK: 84.73% | Val PCK: 88.81% | Val OKS: 0.2487 | Val Depth MAE: 1.8124
Epoch [06/15] | Train PCK: 86.57% | Val PCK: 91.02% | Val OKS: 0.2368 | Val Depth MAE: 2.0189
Epoch [07/15] | Train PCK: 88.23% | Val PCK: 92.15% | Val OKS: 0.2360 | Val Depth MAE: 1.8232
Epoch [08/15] | Train PCK: 89.34% | Val PCK: 92.35% | Val OKS: 0.3146 | Val Depth MAE: 1.7172
Epoch [09/15] | Train PCK: 89.88% | Val PCK: 93.04% | Val OKS: 

KeyboardInterrupt: 